In [ ]:
# osm_match_schools_batch_fixed.py

import os
import re
import time
import pandas as pd
import requests
from difflib import SequenceMatcher
from tqdm import tqdm

# ===================== НАСТРОЙКИ =====================
IN_CSV  = "kindergartens_almaty_private.csv"
OUT_CSV = "kindergartens_almaty_private_osm_matched.csv"
OSM_DUMP_CSV = "almaty_osm_schools_dump.csv"
NOT_FOUND_CSV = "kindergartens_almaty_private_osm_not_found.csv"

# bbox Алматы
BBOX = (43.10, 76.70, 43.45, 77.15)

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

HEADERS = {"User-Agent": "almaty-osm-school-matcher/2.0"}

ALMATY_LAT = 43.2389
ALMATY_LON = 76.8897

# ===================== OSM =====================
def download_osm_schools(bbox):
    south, west, north, east = bbox

    q = f"""
    [out:json][timeout:180];
    (
        node["amenity"="kindergarten"]({south},{west},{north},{east});
        way["amenity"="kindergarten"]({south},{west},{north},{east});
        relation["amenity"="kindergarten"]({south},{west},{north},{east});

        // иногда размечены как building
        node["building"="kindergarten"]({south},{west},{north},{east});
        way["building"="kindergarten"]({south},{west},{north},{east});
        relation["building"="kindergarten"]({south},{west},{north},{east});
    );
    out center tags;
    """

    for base in OVERPASS_URLS:
        for attempt in range(5):
            try:
                r = requests.post(base, data=q.encode("utf-8"), headers=HEADERS, timeout=240)
                if r.status_code in (429, 502, 503, 504):
                    time.sleep(2 * (attempt + 1))
                    continue

                r.raise_for_status()
                js = r.json()

                rows = []
                for e in js.get("elements", []):
                    tags = e.get("tags", {}) or {}

                    if e["type"] == "node":
                        lat, lon = e.get("lat"), e.get("lon")
                    else:
                        c = e.get("center") or {}
                        lat, lon = c.get("lat"), c.get("lon")

                    if lat is None or lon is None:
                        continue

                    rows.append({
                        "osm_type": e["type"],
                        "osm_id": e["id"],
                        "name": tags.get("name"),
                        "name_ru": tags.get("name:ru"),
                        "lat": lat,
                        "lon": lon,
                    })

                return pd.DataFrame(rows)

            except Exception:
                time.sleep(2)

    raise RuntimeError("Overpass failed")

# ===================== MATCHING =====================
def norm(s):
    if s is None:
        return ""
    s = str(s).lower()
    s = s.replace("ё", "е").replace("№", " no ")
    s = re.sub(r"[^0-9a-zа-я\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def extract_number(s):
    s = norm(s)
    m = re.search(r"\bno\s*(\d+)", s)
    if m:
        return m.group(1)
    m = re.search(r"школ[а-я]*\s*(\d+)", s)
    if m:
        return m.group(1)
    return None

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def best_match(name, osm):
    target = norm(name)
    num = extract_number(name)

    cand = osm
    if num:
        tmp = osm[osm["name_norm"].str.contains(num, na=False)]
        if len(tmp) > 0:
            cand = tmp

    exact = cand[cand["name_norm"] == target]
    if len(exact) > 0:
        r = exact.iloc[0]
        return r, "exact", 1.0

    best = None
    best_score = 0

    for _, r in cand.iterrows():
        sc = similarity(target, r["name_norm"])
        if sc > best_score:
            best_score = sc
            best = r

    if best_score < (0.62 if num else 0.72):
        return None, None, None

    return best, "fuzzy", best_score

# ===================== GEOCODERS =====================
def in_bbox(lat, lon):
    return 43.10 <= lat <= 43.45 and 76.70 <= lon <= 77.15

photon_cache = {}
def photon_geocode(q):
    if q in photon_cache:
        return photon_cache[q]

    url = "https://photon.komoot.io/api"
    try:
        r = requests.get(url, params={"q": q, "limit": 1}, timeout=10)
        js = r.json()
        feats = js.get("features", [])
        if feats:
            lon, lat = feats[0]["geometry"]["coordinates"]
            if in_bbox(lat, lon):
                photon_cache[q] = (lat, lon)
                return lat, lon
    except:
        pass

    photon_cache[q] = (None, None)
    return None, None

# ===================== MAIN =====================
def main():
    df = pd.read_csv(IN_CSV)

    # OSM
    if os.path.exists(OSM_DUMP_CSV):
        osm = pd.read_csv(OSM_DUMP_CSV)
    else:
        osm = download_osm_schools(BBOX)
        osm.to_csv(OSM_DUMP_CSV, index=False)

    osm["name_best"] = osm["name_ru"].fillna(osm["name"]).fillna("")
    osm["name_norm"] = osm["name_best"].apply(norm)

    # OUTPUT
    out = df.copy()
    out["lat"] = None
    out["lon"] = None
    out["method"] = None
    out["score"] = None

    # 1. OSM matching
    for i, row in tqdm(out.iterrows(), total=len(out)):
        res, method, score = best_match(row["name"], osm)
        if res is not None:
            out.at[i, "lat"] = res["lat"]
            out.at[i, "lon"] = res["lon"]
            out.at[i, "method"] = method
            out.at[i, "score"] = score

    # 2. Photon fallback
    for i in tqdm(out[out["lat"].isna()].index):
        addr = str(out.at[i, "address"])
        lat, lon = photon_geocode(addr + ", Алматы")

        if lat:
            out.at[i, "lat"] = lat
            out.at[i, "lon"] = lon
            out.at[i, "method"] = "photon"

        time.sleep(0.2)

    # 3. Nominatim fallback
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

    geo = Nominatim(user_agent="almaty-final")
    geocode = RateLimiter(geo.geocode, min_delay_seconds=1)

    for i in tqdm(out[out["lat"].isna()].index):
        addr = str(out.at[i, "address"])
        try:
            loc = geocode(addr + ", Алматы, Казахстан")
            if loc and in_bbox(loc.latitude, loc.longitude):
                out.at[i, "lat"] = loc.latitude
                out.at[i, "lon"] = loc.longitude
                out.at[i, "method"] = "nominatim"
        except:
            pass

    # confidence
    def conf(row):
        if row["method"] == "exact":
            return "high"
        if row["method"] == "fuzzy" and row["score"] > 0.8:
            return "high"
        if row["method"] in ["photon", "nominatim"]:
            return "medium"
        return "low"

    out["confidence"] = out.apply(conf, axis=1)

    out.to_csv(OUT_CSV, index=False)
    out[out["lat"].isna()].to_csv(NOT_FOUND_CSV, index=False)

    print("DONE")
    print("Found:", out["lat"].notna().sum())
    print("Not found:", out["lat"].isna().sum())

if __name__ == "__main__":
    main()

In [1]:
import pandas as pd
df=pd.read_csv("kindergartens_almaty_private_osm_matched_final11.csv")

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 723 entries, 0 to 722
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          723 non-null    object 
 1   address       723 non-null    object 
 2   phone         721 non-null    object 
 3   type          721 non-null    object 
 4   email         720 non-null    object 
 5   lat           723 non-null    float64
 6   lon           723 non-null    float64
 7   method        571 non-null    object 
 8   score         266 non-null    float64
 9   confidence    722 non-null    object 
 10  match_method  99 non-null     object 
 11  money         723 non-null    object 
dtypes: float64(3), object(9)
memory usage: 67.9+ KB


In [3]:
df.drop(columns=["method", "score", 'confidence', 'match_method'], inplace=True)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 723 entries, 0 to 722
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   name     723 non-null    object 
 1   address  723 non-null    object 
 2   phone    721 non-null    object 
 3   type     721 non-null    object 
 4   email    720 non-null    object 
 5   lat      723 non-null    float64
 6   lon      723 non-null    float64
 7   money    723 non-null    object 
dtypes: float64(2), object(6)
memory usage: 45.3+ KB


In [11]:
df[df['type'].isna()]



,name,address,phone,type,email,lat,lon,money


In [ ]:
df['type'].fillna("ясли детский садб", inplace=True)

In [4]:
import pandas as pd
df=pd.read_csv("kindergartens_almaty_osm_matched_final11.csv")

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          198 non-null    object 
 1   address       198 non-null    object 
 2   phone         198 non-null    object 
 3   type          195 non-null    object 
 4   email         198 non-null    object 
 5   lat           198 non-null    float64
 6   lon           198 non-null    float64
 7   method        176 non-null    object 
 8   score         161 non-null    float64
 9   confidence    198 non-null    object 
 10  match_method  21 non-null     object 
 11  money         198 non-null    object 
dtypes: float64(3), object(9)
memory usage: 18.7+ KB


In [6]:
dups = df[df.duplicated(subset=["lat","lon"], keep=False)].sort_values(["lat","lon"])
print(dups[["name","address","lat","lon","method"]].to_string())


                                                                         name                           address        lat        lon method
59                                                       КГКП «Ясли-сад №132»          ул. Қарасай батыр, 200/1  43.249377  76.897412  fuzzy
167                                                       КГКП "Ясли-сад №32"                  ул. Магнитная, 4  43.249377  76.897412  fuzzy
6                                                        КГКП «Ясли-сад №186»                  мкр. Аксай-4, 64  43.249681  76.930603  fuzzy
18                                                       КГКП "Ясли-сад №138"                        мкр. 7, 40  43.249681  76.930603  fuzzy
28                                                       КГКП "Ясли-сад №158"                  мкр. Баянаул, 19  43.249681  76.930603  fuzzy
103                                                      КГКП "Ясли-сад №108"                ул. Тимирязева, 25  43.249681  76.930603  fuzzy
161          

In [38]:
import pandas as pd


dup_mask = df.duplicated(subset=["lat","lon"], keep=False)
dup_mask = dup_mask & df["lat"].notna()
dups = df[dup_mask].copy()

print("=== ОДИНАКОВЫЙ АДРЕС ===\n")
same_addr_count = 0
for (lat, lon), group in dups.groupby(["lat","lon"]):
    if group["address"].nunique() == 1:
        same_addr_count += 1
        print(f"Адрес: {group['address'].iloc[0]}")
        for _, row in group.iterrows():
            print(f"  - {row['name']}")
        print()

print(f"Всего групп с одинаковым адресом: {same_addr_count}")
print()

print("=== РАЗНЫЕ АДРЕСА (промах геокодера) ===\n")
diff_addr_count = 0
for (lat, lon), group in dups.groupby(["lat","lon"]):
    if group["address"].nunique() > 1:
        diff_addr_count += 1
        print(f"Точка: {lat}, {lon}")
        for _, row in group.iterrows():
            print(f"  - {row['name']} | {row['address']}")
        print()
        

print(f"Всего групп с разными адресами: {diff_addr_count}")


=== ОДИНАКОВЫЙ АДРЕС ===

Всего групп с одинаковым адресом: 0

=== РАЗНЫЕ АДРЕСА (промах геокодера) ===

Всего групп с разными адресами: 0


In [37]:
df.loc[df['name']=='КГКП "Ясли-сад №57"', 'lat'] = 43.227477
df.loc[df['name']=='КГКП "Ясли-сад №57"', 'lon'] = 76.839618

df.loc[df['name']=='КГКП "Ясли-сад №113"', 'lat'] = 43.223778
df.loc[df['name']=='КГКП "Ясли-сад №113"', 'lon'] = 76.892909

df.loc[df['name']=='КГКП "Ясли-сад №112"', 'lat'] = 43.223785
df.loc[df['name']=='КГКП "Ясли-сад №112"', 'lon'] = 76.894109

df.loc[df['name']=='КГКП "Ясли-сад №71"', 'lat'] = 43.230127
df.loc[df['name']=='КГКП "Ясли-сад №71"', 'lon'] = 76.90768

In [34]:
df[df["name"].str.contains("123", na=False)]


,name,address,phone,type,email,lat,lon,method,score,confidence,match_method,money
157,"КГУ ""Специальный детский сад №123""","мкр. Алтай 1, 2а","87017397081, 87272512137",специальный детский сад,2322743@mail.ru,43.343179,76.986402,NaN,NaN,low,nominatim_clean,no


In [39]:
df.to_csv("kindergartens_almaty_osm_matched_final11.csv", index=False)

In [40]:
df1=pd.read_csv('kindergartens_almaty_private_osm_matched_final11.csv')

In [41]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          198 non-null    object 
 1   address       198 non-null    object 
 2   phone         198 non-null    object 
 3   type          195 non-null    object 
 4   email         198 non-null    object 
 5   lat           198 non-null    float64
 6   lon           198 non-null    float64
 7   method        176 non-null    object 
 8   score         161 non-null    float64
 9   confidence    198 non-null    object 
 10  match_method  21 non-null     object 
 11  money         198 non-null    object 
dtypes: float64(3), object(9)
memory usage: 18.7+ KB


In [44]:
df=pd.read_csv('kindergartens_almaty_osm_matched_final.csv')
df1=pd.read_csv('kindergartens_almaty_private_osm_matched_final.csv')

df_all = pd.concat([df, df1], ignore_index=True)

In [45]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 924 entries, 0 to 923
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          924 non-null    object 
 1   address       924 non-null    object 
 2   phone         918 non-null    object 
 3   type          919 non-null    object 
 4   email         917 non-null    object 
 5   lat           924 non-null    float64
 6   lon           924 non-null    float64
 7   method        753 non-null    object 
 8   score         426 non-null    float64
 9   confidence    919 non-null    object 
 10  match_method  119 non-null    object 
 11  money         924 non-null    object 
dtypes: float64(3), object(9)
memory usage: 86.8+ KB


In [46]:
df_all.drop(columns=["method", "score", 'confidence', 'match_method'], inplace=True)

In [47]:
df_all.to_csv('kindergardens_result.csv', index=False)